# 04. Logisch schließen

Wie löst du ein Sudoku? Sicher nicht, indem du in das erste leere Feld eine 1 schreibst und schaust, ob es gut geht. Du
suchst ein Feld, in das **nur eine** Ziffer passt, trägst sie ein und machst weiter. Erst wenn nichts mehr zwingend ist,
probierst du etwas aus.

Genau so arbeiten gute Programme für Rätselaufgaben. Das Backtracking aus [Notebook 02](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/02_backtracking.ipynb)
bricht ab, wenn etwas **schon falsch** ist. Logisches Schließen (englisch *constraint propagation*) geht weiter: Es trägt ein,
was **zwingend folgt**. Danach bleibt oft fast nichts mehr zu raten.

In diesem Notebook lernst du:

- für ein Feld die **Kandidaten** zu bestimmen (was passt hier noch?),
- **Regeln** anzuwenden, die zwingende Folgerungen liefern, und das zu wiederholen, **bis sich nichts mehr ändert**,
- **Widersprüche** zu erkennen,
- erst dann zu **raten**, nach jedem Raten wieder zu schließen und die Aufrufe zu zählen,
- zu prüfen, ob die Lösung **eindeutig** ist,
- mit **drei Zuständen** je Feld zu arbeiten: ja, nein, noch offen.

In der Wettbewerbs-Ecke rekonstruierst du Figuren aus ihren Zeilen- und Spaltensummen wie der Biberazzo Azze.

## 1. Mini-Sudoku: Was passt hier noch?

Ein Mini-Sudoku hat 4 × 4 Felder. In jede Zeile, jede Spalte und jeden der vier 2 × 2-Blöcke kommen die Ziffern 1 bis 4
je genau einmal. Wir speichern es als Liste von Zeilen, eine **0** steht für ein leeres Feld:

    2 . . .
    . . 4 .
    . . . 1
    3 . . .

Die **Kandidaten** eines leeren Feldes sind die Ziffern, die weder in seiner Zeile noch in seiner Spalte noch in seinem
Block stehen. Der Block von Feld `(r, c)` beginnt bei Zeile `r - r % 2` und Spalte `c - c % 2`. Damit dasselbe Programm auch
für das normale 9 × 9-Sudoku mit 3 × 3-Blöcken funktioniert, rechnen wir die Blockgröße `b` aus: Sie ist die Wurzel aus der
Zahl der Zeilen.

`menge.discard(x)` entfernt `x` aus einer Menge und beschwert sich nicht, wenn `x` gar nicht darin ist (anders als
`remove`). So kann man auch die 0 der leeren Felder „entfernen“.

#### Aufgabe 1.1

Schreibe `kandidaten(gitter, r, c)`, die die Menge der Kandidaten von Feld `(r, c)` zurückgibt. `zeige` ist fertig.

**Erwartete Ausgabe:**
```
2 . . . 
. . 4 . 
. . . 1 
3 . . . 
[1, 3, 4] [1] [3]
```

In [ ]:
def zeige(gitter):
    for zeile in gitter:
        text = ''
        for ziffer in zeile:
            if ziffer == 0:
                text = text + '. '
            else:
                text = text + str(ziffer) + ' '
        print(text)

def kandidaten(gitter, r, c):
    n = len(gitter)
    b = int(n ** 0.5)                  # Blockgröße: 2 bei 4 × 4, 3 bei 9 × 9
    moeglich = set(range(1, n + 1))
    # dein Code
    return moeglich

sudoku_a = [[2, 0, 0, 0], [0, 0, 4, 0], [0, 0, 0, 1], [3, 0, 0, 0]]
zeige(sudoku_a)
print(sorted(kandidaten(sudoku_a, 0, 1)), sorted(kandidaten(sudoku_a, 1, 0)), sorted(kandidaten(sudoku_a, 0, 3)))

<details>
<summary>Tipp anzeigen</summary>

Eine Schleife `for k in range(n):` entfernt `gitter[r][k]` (Zeile) und `gitter[k][c]` (Spalte). Für den Block zwei Schleifen
von `r0 = r - r % b` bis `r0 + b` und von `c0 = c - c % b` bis `c0 + b`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def zeige(gitter):
    for zeile in gitter:
        text = ''
        for ziffer in zeile:
            if ziffer == 0:
                text = text + '. '
            else:
                text = text + str(ziffer) + ' '
        print(text)

def kandidaten(gitter, r, c):
    n = len(gitter)
    b = int(n ** 0.5)                  # Blockgröße: 2 bei 4 × 4, 3 bei 9 × 9
    moeglich = set(range(1, n + 1))
    for k in range(n):
        moeglich.discard(gitter[r][k])     # Zeile
        moeglich.discard(gitter[k][c])     # Spalte
    r0 = r - r % b
    c0 = c - c % b
    for i in range(r0, r0 + b):            # Block
        for j in range(c0, c0 + b):
            moeglich.discard(gitter[i][j])
    return moeglich

sudoku_a = [[2, 0, 0, 0], [0, 0, 4, 0], [0, 0, 0, 1], [3, 0, 0, 0]]
zeige(sudoku_a)
print(sorted(kandidaten(sudoku_a, 0, 1)), sorted(kandidaten(sudoku_a, 1, 0)), sorted(kandidaten(sudoku_a, 0, 3)))
```

In Feld `(1, 0)` passt nur die 1: Die 4 steht in derselben Zeile, die 2 und die 3 stehen in derselben Spalte. In Feld `(0, 3)`
passt nur die 3. Diese Ziffern sind **zwingend**.

</details>

### Regel 1: Nur ein Kandidat

Hat ein leeres Feld nur **einen** Kandidaten, muss diese Ziffer hinein. Hat es **keinen**, ist das Sudoku **widersprüchlich**:
So kann es keine Lösung geben.

Jede eingetragene Ziffer nimmt anderen Feldern Kandidaten weg. Darum geht man alle Felder durch, und wenn sich dabei etwas
geändert hat, **noch einmal**, bis ein ganzer Durchgang nichts mehr ändert. Dafür eignet sich eine `while`-Schleife mit einer
Variablen `geaendert`.

#### Aufgabe 1.2

Schreibe `schliesse(gitter)`, die Regel 1 anwendet, bis sich nichts mehr ändert. Sie verändert `gitter` und gibt `False`
zurück, wenn ein Feld keinen Kandidaten mehr hat, sonst `True`. Löse damit `sudoku_a`.

**Erwartete Ausgabe:**
```
True
2 4 1 3 
1 3 4 2 
4 2 3 1 
3 1 2 4
```

In [ ]:
def schliesse(gitter):
    n = len(gitter)
    geaendert = True
    while geaendert:
        geaendert = False
        # dein Code
    return True

loesung_a = [[2, 0, 0, 0], [0, 0, 4, 0], [0, 0, 0, 1], [3, 0, 0, 0]]
print(schliesse(loesung_a))
zeige(loesung_a)

<details>
<summary>Tipp anzeigen</summary>

Zwei Schleifen über alle Felder. Bei einem leeren Feld die Kandidaten bestimmen: keiner → `return False`, genau einer →
eintragen (`moeglich.pop()` holt das einzige Element aus der Menge) und `geaendert = True`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def schliesse(gitter):
    n = len(gitter)
    geaendert = True
    while geaendert:
        geaendert = False
        for r in range(n):
            for c in range(n):
                if gitter[r][c] == 0:
                    moeglich = kandidaten(gitter, r, c)
                    if len(moeglich) == 0:
                        return False                    # Widerspruch
                    if len(moeglich) == 1:
                        gitter[r][c] = moeglich.pop()   # zwingend
                        geaendert = True
    return True

loesung_a = [[2, 0, 0, 0], [0, 0, 4, 0], [0, 0, 0, 1], [3, 0, 0, 0]]
print(schliesse(loesung_a))
zeige(loesung_a)
```

Obwohl nur vier Ziffern vorgegeben sind, ist das ganze Sudoku bestimmt, und zwar ohne ein einziges Mal zu raten.

</details>

#### Frage 1.3

1. Warum darf `schliesse` eine Ziffer eintragen, ohne zu wissen, ob das Sudoku überhaupt lösbar ist?
2. Warum muss die `while`-Schleife wiederholen, statt nur einmal über alle Felder zu gehen?
3. Endet die `while`-Schleife sicher?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Wenn es eine Lösung gibt, steht in diesem Feld eine Ziffer, die nicht in Zeile, Spalte oder Block vorkommt, also ein
   Kandidat. Gibt es nur einen, muss es dieser sein. Das Eintragen schließt also **keine Lösung aus**. Gibt es gar keine
   Lösung, ist das Eintragen auch nicht schlimm: Irgendwann bleibt ein Feld ohne Kandidaten, und das Programm meldet den
   Widerspruch.
2. Eine eingetragene Ziffer kann einem Feld, das schon geprüft wurde, den vorletzten Kandidaten wegnehmen. Das erkennt erst
   der nächste Durchgang.
3. Ja. Jeder Durchgang außer dem letzten trägt mindestens eine Ziffer ein, und es gibt nur `n · n` Felder. Ein Durchgang
   kostet `n · n` Kandidatenprüfungen, also gibt es höchstens etwa n⁴ solcher Prüfungen: Das ist schnell.

</details>

## 2. Eine zweite Regel

Bei einem echten 9 × 9-Sudoku reicht Regel 1 oft nicht. Die Datei `daten/sudoku_mittel.txt` enthält ein Sudoku, das
`schliesse` nicht ganz löst. Eine zweite Regel hilft:

**Regel 2: Nur ein Platz.** Betrachte eine **Einheit** (eine Zeile, eine Spalte oder einen Block) und eine Ziffer, die dort
noch fehlt. Ist sie nur in **einem** leeren Feld der Einheit Kandidat, muss sie dort hin. Ist sie in **keinem** Feld
Kandidat, ist das Sudoku widersprüchlich.

Regel 1 fragt „welche Ziffer passt in dieses Feld?“, Regel 2 fragt „wohin passt diese Ziffer?“. Die Funktion `einheiten(n)`
liefert alle Einheiten als Listen von Feldern, `lies_sudoku` liest eine Datei mit `.` für leere Felder.

#### Aufgabe 2.1

Schreibe `schliesse2(gitter)`. In einer `while`-Schleife wendet sie zuerst `schliesse` an (Regel 1) und dann Regel 2 auf alle
Einheiten und Ziffern. Hat Regel 2 etwas eingetragen, beginnt die Schleife von vorn. Vergleiche für `sudoku_mittel.txt`, wie
viele Felder nach `schliesse` und nach `schliesse2` noch leer sind.

**Erwartete Ausgabe:**
```
vorher 57 leere Felder
Regel 1: True 54 leere Felder
Regel 1 und 2: True 0 leere Felder
6 3 9 2 5 1 7 4 8 
4 5 8 6 3 7 1 9 2 
7 1 2 9 4 8 3 6 5 
3 9 4 8 2 5 6 1 7 
1 7 5 3 6 9 8 2 4 
8 2 6 7 1 4 9 5 3 
9 8 1 4 7 2 5 3 6 
2 6 7 5 9 3 4 8 1 
5 4 3 1 8 6 2 7 9
```

In [ ]:
def einheiten(n):
    b = int(n ** 0.5)
    alle = []
    for r in range(n):
        zeile = []
        for c in range(n):
            zeile.append((r, c))
        alle.append(zeile)
    for c in range(n):
        spalte = []
        for r in range(n):
            spalte.append((r, c))
        alle.append(spalte)
    for r0 in range(0, n, b):
        for c0 in range(0, n, b):
            block = []
            for r in range(r0, r0 + b):
                for c in range(c0, c0 + b):
                    block.append((r, c))
            alle.append(block)
    return alle

def lies_sudoku(dateiname):
    gitter = []
    for zeile in open(dateiname):
        zeile = zeile.strip()
        if zeile != '':
            ziffern = []
            for zeichen in zeile:
                if zeichen == '.':
                    ziffern.append(0)
                else:
                    ziffern.append(int(zeichen))
            gitter.append(ziffern)
    return gitter

def leere(gitter):
    anzahl = 0
    for zeile in gitter:
        anzahl = anzahl + zeile.count(0)
    return anzahl

def schliesse2(gitter):
    n = len(gitter)
    geaendert = True
    while geaendert:
        if not schliesse(gitter):
            return False
        geaendert = False
        # dein Code
    return True

mittel1 = lies_sudoku('daten/sudoku_mittel.txt')
mittel2 = lies_sudoku('daten/sudoku_mittel.txt')
print('vorher', leere(mittel1), 'leere Felder')
print('Regel 1:', schliesse(mittel1), leere(mittel1), 'leere Felder')
print('Regel 1 und 2:', schliesse2(mittel2), leere(mittel2), 'leere Felder')
zeige(mittel2)

<details>
<summary>Tipp anzeigen</summary>

Für jede Einheit und jede Ziffer: Sammle in `plaetze` die leeren Felder der Einheit, in denen die Ziffer Kandidat ist, und
merke dir in `schon_da`, ob sie dort schon steht. Steht sie nicht da: kein Platz → `return False`, genau ein Platz →
eintragen und `geaendert = True`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def einheiten(n):
    b = int(n ** 0.5)
    alle = []
    for r in range(n):
        zeile = []
        for c in range(n):
            zeile.append((r, c))
        alle.append(zeile)
    for c in range(n):
        spalte = []
        for r in range(n):
            spalte.append((r, c))
        alle.append(spalte)
    for r0 in range(0, n, b):
        for c0 in range(0, n, b):
            block = []
            for r in range(r0, r0 + b):
                for c in range(c0, c0 + b):
                    block.append((r, c))
            alle.append(block)
    return alle

def lies_sudoku(dateiname):
    gitter = []
    for zeile in open(dateiname):
        zeile = zeile.strip()
        if zeile != '':
            ziffern = []
            for zeichen in zeile:
                if zeichen == '.':
                    ziffern.append(0)
                else:
                    ziffern.append(int(zeichen))
            gitter.append(ziffern)
    return gitter

def leere(gitter):
    anzahl = 0
    for zeile in gitter:
        anzahl = anzahl + zeile.count(0)
    return anzahl

def schliesse2(gitter):
    n = len(gitter)
    geaendert = True
    while geaendert:
        if not schliesse(gitter):                       # erst Regel 1
            return False
        geaendert = False
        for einheit in einheiten(n):                    # dann Regel 2
            for ziffer in range(1, n + 1):
                plaetze = []
                schon_da = False
                for r, c in einheit:
                    if gitter[r][c] == ziffer:
                        schon_da = True
                    elif gitter[r][c] == 0 and ziffer in kandidaten(gitter, r, c):
                        plaetze.append((r, c))
                if not schon_da:
                    if len(plaetze) == 0:
                        return False                    # Ziffer passt nirgends hin
                    if len(plaetze) == 1:
                        r, c = plaetze[0]
                        gitter[r][c] = ziffer           # nur ein Platz
                        geaendert = True
    return True

mittel1 = lies_sudoku('daten/sudoku_mittel.txt')
mittel2 = lies_sudoku('daten/sudoku_mittel.txt')
print('vorher', leere(mittel1), 'leere Felder')
print('Regel 1:', schliesse(mittel1), leere(mittel1), 'leere Felder')
print('Regel 1 und 2:', schliesse2(mittel2), leere(mittel2), 'leere Felder')
zeige(mittel2)
```

Regel 1 trägt hier nur 3 Ziffern ein, dann bleiben 54 Felder leer. Mit Regel 2 ist das Sudoku ganz gelöst, ohne zu raten.

</details>

## 3. Raten und wieder schließen

Manchmal helfen auch beide Regeln nicht weiter. Dann wird geraten, aber geschickt:

1. **Schließen.** Gibt es einen Widerspruch, war ein früheres Raten falsch: aufgeben (`None`).
2. Ist kein Feld mehr leer, ist das Sudoku gelöst.
3. Sonst das erste leere Feld nehmen und jeden seiner Kandidaten **ausprobieren**: eintragen und die Funktion selbst
   aufrufen. Die erste Lösung, die zurückkommt, wird weitergegeben.

Anders als in Notebook 02 nehmen wir hier nichts zurück, sondern arbeiten in jedem Aufruf mit einer **Kopie** des Gitters.
Das Schließen trägt nämlich auf einmal viele Ziffern ein, und die alle wieder zu entfernen wäre mühsam. Die Kopie eines
Sudokus ist dagegen billig. Achtung: Eine Liste von Listen kopiert man **Zeile für Zeile** (`kopiere` ist fertig). Warum,
siehst du in Abschnitt 5.

Mit dem Parameter `regeln` schalten wir um: 0 heißt nur raten (Backtracking wie in Notebook 02, aber nur mit Kandidaten),
1 heißt vorher `schliesse`, 2 heißt vorher `schliesse2`.

#### Aufgabe 3.1

Schreibe `loese(gitter, regeln)`. Löse damit die drei 9 × 9-Sudokus aus `daten/` und vergleiche die Aufrufe für
`regeln` = 0, 1 und 2. Zeige die Lösung des schweren Sudokus.

**Erwartete Ausgabe:**
```
leicht : 54 leere Felder, Aufrufe mit regeln = 0, 1, 2: [931, 1, 1]
mittel : 57 leere Felder, Aufrufe mit regeln = 0, 1, 2: [8306, 19, 1]
schwer : 57 leere Felder, Aufrufe mit regeln = 0, 1, 2: [47790, 261, 11]
6 5 8 4 9 2 7 3 1 
9 7 4 3 5 1 2 8 6 
2 3 1 7 8 6 4 5 9 
8 4 5 9 3 7 1 6 2 
1 6 9 5 2 4 8 7 3 
3 2 7 6 1 8 5 9 4 
5 1 3 8 4 9 6 2 7 
7 8 2 1 6 3 9 4 5 
4 9 6 2 7 5 3 1 8
```

In [ ]:
def kopiere(gitter):
    neu = []
    for zeile in gitter:
        neu.append(list(zeile))
    return neu

aufrufe = [0]

def loese(gitter, regeln):
    aufrufe[0] = aufrufe[0] + 1
    gitter = kopiere(gitter)
    # dein Code
    return None

for name in ['leicht', 'mittel', 'schwer']:
    sudoku = lies_sudoku('daten/sudoku_' + name + '.txt')
    zahlen = []
    for regeln in [0, 1, 2]:
        aufrufe[0] = 0
        loesung = loese(sudoku, regeln)
        zahlen.append(aufrufe[0])
    print(name, ':', leere(sudoku), 'leere Felder, Aufrufe mit regeln = 0, 1, 2:', zahlen)
if loesung is not None:
    zeige(loesung)

<details>
<summary>Tipp anzeigen</summary>

Nach dem Kopieren: Bei `regeln == 1` mit `schliesse`, bei `regeln == 2` mit `schliesse2` schließen und bei `False` `None`
zurückgeben. Dann das erste leere Feld suchen, für jeden Kandidaten eintragen und `loese(gitter, regeln)` aufrufen. Ist das
Ergebnis nicht `None`, sofort zurückgeben. Nach der Kandidatenschleife `return None`. Ist kein Feld leer, `return gitter`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def kopiere(gitter):
    neu = []
    for zeile in gitter:
        neu.append(list(zeile))
    return neu

aufrufe = [0]

def loese(gitter, regeln):
    aufrufe[0] = aufrufe[0] + 1
    gitter = kopiere(gitter)
    if regeln == 1 and not schliesse(gitter):
        return None
    if regeln == 2 and not schliesse2(gitter):
        return None
    n = len(gitter)
    for r in range(n):
        for c in range(n):
            if gitter[r][c] == 0:                       # erstes leeres Feld
                for ziffer in sorted(kandidaten(gitter, r, c)):
                    gitter[r][c] = ziffer               # raten
                    loesung = loese(gitter, regeln)
                    if loesung is not None:
                        return loesung
                return None                             # kein Kandidat führt zum Ziel
    return gitter                                       # kein Feld mehr leer: gelöst

for name in ['leicht', 'mittel', 'schwer']:
    sudoku = lies_sudoku('daten/sudoku_' + name + '.txt')
    zahlen = []
    for regeln in [0, 1, 2]:
        aufrufe[0] = 0
        loesung = loese(sudoku, regeln)
        zahlen.append(aufrufe[0])
    print(name, ':', leere(sudoku), 'leere Felder, Aufrufe mit regeln = 0, 1, 2:', zahlen)
if loesung is not None:
    zeige(loesung)
```

Ein Aufruf heißt: Es musste nicht geraten werden. Beim leichten Sudoku reicht dafür Regel 1, beim mittleren braucht man
Regel 2. Beim schweren muss man auch mit beiden Regeln raten, aber nur 11 Aufrufe statt 47 790 ohne Schließen. Ein Aufruf
mit Schließen dauert zwar länger, aber der Unterschied ist so groß, dass es sich hier deutlich lohnt.

</details>

## 4. Ist die Lösung eindeutig?

Ein gutes Sudoku hat genau **eine** Lösung. Ob das so ist, findet `loese` nicht heraus, denn es hört bei der ersten Lösung
auf. Dafür muss man weitersuchen, aber nicht unbedingt alle Lösungen finden: Sobald es **zwei** gibt, weiß man, dass die
Lösung nicht eindeutig ist. Darum sammeln wir Lösungen nur bis zu einer **Grenze**.

Das Mini-Sudoku `sudoku_b` hat ebenfalls vier Vorgaben, aber hier lässt `schliesse2` vier Felder offen.

#### Aufgabe 4.1

Schreibe `sammle(gitter, grenze, loesungen)`, die wie `loese` mit `schliesse2` arbeitet, aber jede Lösung an `loesungen`
anhängt und aufhört, sobald `grenze` Lösungen gefunden sind. Prüfe `sudoku_a`, `sudoku_b` und die drei 9 × 9-Sudokus.

**Erwartete Ausgabe:**
```
sudoku_a : 1 Lösung(en)
2 4 1 3 
1 3 4 2 
4 2 3 1 
3 1 2 4 

sudoku_b : 2 Lösung(en)
2 4 1 3 
1 3 4 2 
4 2 3 1 
3 1 2 4 

2 4 1 3 
3 1 4 2 
4 2 3 1 
1 3 2 4 

leicht : 1 Lösung(en)
mittel : 1 Lösung(en)
schwer : 1 Lösung(en)
```

In [ ]:
def sammle(gitter, grenze, loesungen):
    gitter = kopiere(gitter)
    # dein Code

sudoku_b = [[0, 0, 1, 0], [0, 0, 0, 2], [4, 2, 0, 0], [0, 0, 0, 0]]
for name, sudoku in [('a', sudoku_a), ('b', sudoku_b)]:
    loesungen_mini = []
    sammle(sudoku, 2, loesungen_mini)
    print('sudoku_' + name, ':', len(loesungen_mini), 'Lösung(en)')
    for loesung in loesungen_mini:
        zeige(loesung)
        print()
for name in ['leicht', 'mittel', 'schwer']:
    loesungen_9 = []
    sammle(lies_sudoku('daten/sudoku_' + name + '.txt'), 2, loesungen_9)
    print(name, ':', len(loesungen_9), 'Lösung(en)')

<details>
<summary>Tipp anzeigen</summary>

Fast wie `loese`: Statt eine Lösung zurückzugeben, hängst du `gitter` an `loesungen` an. Nach jedem Aufruf in der
Kandidatenschleife prüfst du `if len(loesungen) >= grenze: return`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def sammle(gitter, grenze, loesungen):
    gitter = kopiere(gitter)
    if not schliesse2(gitter):
        return
    n = len(gitter)
    for r in range(n):
        for c in range(n):
            if gitter[r][c] == 0:
                for ziffer in sorted(kandidaten(gitter, r, c)):
                    gitter[r][c] = ziffer
                    sammle(gitter, grenze, loesungen)
                    if len(loesungen) >= grenze:
                        return                          # genug gefunden
                return
    loesungen.append(gitter)

sudoku_b = [[0, 0, 1, 0], [0, 0, 0, 2], [4, 2, 0, 0], [0, 0, 0, 0]]
for name, sudoku in [('a', sudoku_a), ('b', sudoku_b)]:
    loesungen_mini = []
    sammle(sudoku, 2, loesungen_mini)
    print('sudoku_' + name, ':', len(loesungen_mini), 'Lösung(en)')
    for loesung in loesungen_mini:
        zeige(loesung)
        print()
for name in ['leicht', 'mittel', 'schwer']:
    loesungen_9 = []
    sammle(lies_sudoku('daten/sudoku_' + name + '.txt'), 2, loesungen_9)
    print(name, ':', len(loesungen_9), 'Lösung(en)')
```

Die beiden Lösungen von `sudoku_b` unterscheiden sich in vier Feldern: In der ersten und zweiten Spalte sind 1 und 3 in den
Zeilen 1 und 3 (von 0 an gezählt) vertauscht. Solche Tauschmuster sind der Grund, warum Sudokus mit zu wenigen Vorgaben mehrdeutig sind.

</details>

#### Frage 4.2

Für die Dokumentation:

1. Warum genügt die Grenze 2, um die Eindeutigkeit zu prüfen?
2. Wie würdest du die Laufzeit von `loese` beschreiben?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Findet `sammle` mit Grenze 2 nur eine Lösung, hat es alle Möglichkeiten durchsucht, ohne eine zweite zu finden: Die
   Lösung ist eindeutig. Findet es zwei, ist sie es nicht. Mehr will man nicht wissen, und das Aufhören spart viel Zeit,
   wenn es sehr viele Lösungen gibt.
2. Im schlimmsten Fall wird für jedes der k leeren Felder jede Ziffer probiert, das sind bis zu 9ᵏ Möglichkeiten,
   also exponentiell. Das logische Schließen ändert daran im schlimmsten Fall nichts, aber bei den Beispielen ist der
   Unterschied riesig (siehe 3.1). In die Dokumentation gehören darum die obere Grenze und die gezählten Aufrufe. Wichtig
   ist auch zu begründen, warum die Regeln **keine Lösung verlieren** (Frage 1.3): Dann ist die Suche vollständig.

</details>

## 5. Finde den Fehler: die halbe Kopie

#### Frage 5.1

Lena schreibt die Kopierfunktion kürzer:

```python
def kopiere_falsch(gitter):
    return list(gitter)
```

und verwendet sie in `loese`. Bei `sudoku_leicht` und `sudoku_mittel` kommt trotzdem die richtige Lösung heraus (mit
beiden Regeln muss dort nicht geraten werden), aber danach ist auch das eingelesene Sudoku ausgefüllt. Beim schweren
Sudoku meldet `loese` „keine Lösung“, obwohl es eine gibt. Was ist falsch?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

`list(gitter)` macht nur eine neue **äußere** Liste. Ihre Elemente sind aber dieselben Zeilenlisten wie vorher. Trägt ein
Aufruf eine Ziffer in eine Zeile ein, steht sie auch im Gitter des Aufrufers. Nach einem falschen Raten bleiben also
falsche Ziffern stehen, und spätere Versuche bauen darauf auf. Bei einer Liste von Listen muss jede Zeile kopiert werden:
`neu.append(list(zeile))`. (Bei einer einfachen Liste von Zahlen reicht `list(...)`, wie in Notebook 02.)

</details>

## Wettbewerbs-Ecke: Bibertomograph

*Angelehnt an Aufgabe 4 „Bibertomograph“ der 1. Runde des 44. Bundeswettbewerbs Informatik (2025/26).*

Auf einer Fläche aus n × n Feldern ist eine Figur aus gefüllten Feldern gelegt. Der Biberazzo Azze sieht sie nicht, kennt aber
für jede Spalte und jede Zeile die **Zahl der gefüllten Felder**. Gesucht ist eine passende Figur. Ist sie nicht die einzig
mögliche, sollen die Felder, die nicht eindeutig sind, ein **Fragezeichen** bekommen.

In der echten Aufgabe kennt Azze auch die Summen entlang der Diagonalen. Wir beginnen mit Zeilen und Spalten. Die Dateien
enthalten in der ersten Zeile `n`, in der zweiten die Summen der Spalten von links nach rechts und in der dritten die der
Zeilen von oben nach unten.

**Drei Zustände.** Jedes Feld ist gefüllt (1), leer (0) oder noch **offen** (`OFFEN = -1`).

**Linien.** Wir behandeln Zeilen und Spalten gleich: Jede **Linie** ist eine Liste von Feldern `(r, c)`, dazu gehört in einer
zweiten Liste `soll` die Zahl der gefüllten Felder. Danach muss das Programm nicht mehr wissen, ob es eine Zeile oder eine
Spalte vor sich hat, und für die Diagonalen braucht man später nur weitere Linien.

#### Aufgabe E.1

Schreibe `baue_linien(n, spalten, zeilen)`, die die Listen `linien` und `soll` zurückgibt: zuerst die Spalten, dann die
Zeilen.

**Erwartete Ausgabe:**
```
[[5], [5, 3, 2, 2, 1], [5, 4, 2, 1, 1]]
10 Linien
[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0)] 5
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)] 5
```

In [ ]:
OFFEN = -1

def lies_tomograph(dateiname):
    zeilen = []
    for zeile in open(dateiname):
        if zeile.strip() != '':
            zahlen = []
            for teil in zeile.split():
                zahlen.append(int(teil))
            zeilen.append(zahlen)
    return zeilen

def baue_linien(n, spalten, zeilen):
    linien = []
    soll = []
    # dein Code
    return linien, soll

daten_treppe = lies_tomograph('daten/raster_treppe.txt')
print(daten_treppe)
linien, soll = baue_linien(daten_treppe[0][0], daten_treppe[1], daten_treppe[2])
print(len(linien), 'Linien')
if len(linien) > 0:
    print(linien[0], soll[0])
    print(linien[5], soll[5])

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
OFFEN = -1

def lies_tomograph(dateiname):
    zeilen = []
    for zeile in open(dateiname):
        if zeile.strip() != '':
            zahlen = []
            for teil in zeile.split():
                zahlen.append(int(teil))
            zeilen.append(zahlen)
    return zeilen

def baue_linien(n, spalten, zeilen):
    linien = []
    soll = []
    for c in range(n):                      # Spalte c
        linie = []
        for r in range(n):
            linie.append((r, c))
        linien.append(linie)
        soll.append(spalten[c])
    for r in range(n):                      # Zeile r
        linie = []
        for c in range(n):
            linie.append((r, c))
        linien.append(linie)
        soll.append(zeilen[r])
    return linien, soll

daten_treppe = lies_tomograph('daten/raster_treppe.txt')
print(daten_treppe)
linien, soll = baue_linien(daten_treppe[0][0], daten_treppe[1], daten_treppe[2])
print(len(linien), 'Linien')
if len(linien) > 0:
    print(linien[0], soll[0])
    print(linien[5], soll[5])
```

</details>

**Die Regeln für eine Linie.** Zähle in der Linie die gefüllten und die offenen Felder. `rest` ist die Zahl der gefüllten
Felder, die noch fehlen.

- Ist `rest` negativ oder größer als die Zahl der offenen Felder: **Widerspruch**.
- Ist `rest == 0`: Alle offenen Felder der Linie sind **leer**.
- Ist `rest` gleich der Zahl der offenen Felder: Alle offenen Felder sind **gefüllt**.

Wie bei `schliesse` wiederholt man das für alle Linien, bis sich nichts mehr ändert.

#### Aufgabe E.2

Schreibe `schliesse_linien(raster, linien, soll)`. Wende sie auf ein ganz offenes Raster für `raster_treppe.txt` an.

**Erwartete Ausgabe:**
```
True
1 1 1 1 1 
1 1 1 1 0 
1 1 0 0 0 
1 0 0 0 0 
1 0 0 0 0
```

In [ ]:
def schliesse_linien(raster, linien, soll):
    geaendert = True
    while geaendert:
        geaendert = False
        # dein Code
    return True

def zeige_raster(raster):
    for zeile in raster:
        text = ''
        for wert in zeile:
            if wert == OFFEN:
                text = text + '. '
            else:
                text = text + str(wert) + ' '
        print(text)

raster = [[OFFEN] * 5 for r in range(5)]
print(schliesse_linien(raster, linien, soll))
zeige_raster(raster)

<details>
<summary>Tipp anzeigen</summary>

Für jede Linie `k`: `gefuellt` zählen und die offenen Felder in einer Liste `offen` sammeln, dann `rest = soll[k] - gefuellt`.
Gibt es offene Felder und ist `rest == 0` oder `rest == len(offen)`, alle offenen Felder auf 0 bzw. 1 setzen und
`geaendert = True`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def schliesse_linien(raster, linien, soll):
    geaendert = True
    while geaendert:
        geaendert = False
        for k in range(len(linien)):
            gefuellt = 0
            offen = []
            for r, c in linien[k]:
                if raster[r][c] == 1:
                    gefuellt = gefuellt + 1
                elif raster[r][c] == OFFEN:
                    offen.append((r, c))
            rest = soll[k] - gefuellt
            if rest < 0 or rest > len(offen):
                return False                            # Widerspruch
            if len(offen) > 0 and (rest == 0 or rest == len(offen)):
                if rest == 0:
                    wert = 0                            # alle offenen leer
                else:
                    wert = 1                            # alle offenen gefüllt
                for r, c in offen:
                    raster[r][c] = wert
                geaendert = True
    return True

def zeige_raster(raster):
    for zeile in raster:
        text = ''
        for wert in zeile:
            if wert == OFFEN:
                text = text + '. '
            else:
                text = text + str(wert) + ' '
        print(text)

raster = [[OFFEN] * 5 for r in range(5)]
print(schliesse_linien(raster, linien, soll))
zeige_raster(raster)
```

`[[OFFEN] * 5 for r in range(5)]` baut fünf **verschiedene** Zeilen. (`[[OFFEN] * 5] * 5` wäre fünfmal **dieselbe** Zeile,
derselbe Fehler wie in Abschnitt 5.) Die Treppe ist allein durch Schließen bestimmt: Zeile 0 und Spalte 0 sind voll, danach
folgt der Rest Schritt für Schritt.

</details>

#### Aufgabe E.3

Schreibe `suche_figur(raster, linien, soll)` wie `loese`: kopieren, schließen, das erste offene Feld erst mit 1, dann mit 0
probieren. Sie gibt eine passende Figur zurück oder `None`. Probiere die drei eigenen Raster.

**Erwartete Ausgabe:**
```
treppe : 1 Aufrufe
1 1 1 1 1 
1 1 1 1 0 
1 1 0 0 0 
1 0 0 0 0 
1 0 0 0 0 
ring : 5 Aufrufe
1 1 0 0 0 
1 1 0 0 0 
0 0 1 1 0 
0 0 1 1 0 
0 0 0 0 0 
unmoeglich : 1 Aufrufe
keine passende Figur
```

In [ ]:
aufrufe = [0]

def suche_figur(raster, linien, soll):
    aufrufe[0] = aufrufe[0] + 1
    raster = kopiere(raster)
    # dein Code
    return None

for name in ['treppe', 'ring', 'unmoeglich']:
    daten = lies_tomograph('daten/raster_' + name + '.txt')
    n = daten[0][0]
    linien, soll = baue_linien(n, daten[1], daten[2])
    aufrufe[0] = 0
    figur = suche_figur([[OFFEN] * n for r in range(n)], linien, soll)
    print(name, ':', aufrufe[0], 'Aufrufe')
    if figur is None:
        print('keine passende Figur')
    else:
        zeige_raster(figur)

<details>
<summary>Tipp anzeigen</summary>

Wie `loese` mit `schliesse_linien` statt `schliesse2` und den Werten `[1, 0]` statt der Kandidaten. `kopiere` aus Abschnitt 3
funktioniert auch hier.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
aufrufe = [0]

def suche_figur(raster, linien, soll):
    aufrufe[0] = aufrufe[0] + 1
    raster = kopiere(raster)
    if not schliesse_linien(raster, linien, soll):
        return None
    n = len(raster)
    for r in range(n):
        for c in range(n):
            if raster[r][c] == OFFEN:
                for wert in [1, 0]:
                    raster[r][c] = wert
                    figur = suche_figur(raster, linien, soll)
                    if figur is not None:
                        return figur
                return None
    return raster

for name in ['treppe', 'ring', 'unmoeglich']:
    daten = lies_tomograph('daten/raster_' + name + '.txt')
    n = daten[0][0]
    linien, soll = baue_linien(n, daten[1], daten[2])
    aufrufe[0] = 0
    figur = suche_figur([[OFFEN] * n for r in range(n)], linien, soll)
    print(name, ':', aufrufe[0], 'Aufrufe')
    if figur is None:
        print('keine passende Figur')
    else:
        zeige_raster(figur)
```

Beim Ring ist das Ergebnis **nicht** der Ring aus der Datei, sondern eine andere Figur mit denselben Summen. Bei
`raster_unmoeglich.txt` verlangt Spalte 0 fünf gefüllte Felder, Zeile 4 aber keines: Das Schließen meldet sofort den
Widerspruch.

</details>

**Fragezeichen.** Ein Feld ist nicht eindeutig, wenn es eine passende Figur gibt, in der es gefüllt ist, **und** eine, in der
es leer ist. Man könnte für jedes Feld und jeden Wert eine eigene Suche starten. Das geht schneller:

- `moeglich[r][c]` ist die Menge der Werte, die Feld `(r, c)` in schon gefundenen Figuren hatte. Am Anfang enthält sie nur
  den Wert aus der ersten Figur.
- Für jedes Feld und jeden Wert, der **noch nicht** in seiner Menge ist: Feld auf diesen Wert setzen (alle anderen offen)
  und eine Figur suchen. Findet man eine, trägt man **alle** ihre Werte in die Mengen ein. Viele spätere Suchen entfallen
  dann.

#### Aufgabe E.4

Schreibe `fragezeichen(n, linien, soll)`. Sie gibt `None` zurück, wenn es keine Figur gibt, sonst eine Liste von Texten mit
`0`, `1` und `?` je Zeile.

**Erwartete Ausgabe:**
```
treppe : 26 Aufrufe
1 1 1 1 1 
1 1 1 1 0 
1 1 0 0 0 
1 0 0 0 0 
1 0 0 0 0 
ring : 62 Aufrufe
? ? ? ? 0 
? ? ? ? 0 
? ? ? ? 0 
? ? ? ? 0 
0 0 0 0 0 
unmoeglich : 1 Aufrufe
keine passende Figur
```

In [ ]:
def fragezeichen(n, linien, soll):
    erste = suche_figur([[OFFEN] * n for r in range(n)], linien, soll)
    if erste is None:
        return None
    moeglich = []
    for r in range(n):
        zeile = []
        for c in range(n):
            zeile.append({erste[r][c]})
        moeglich.append(zeile)
    # dein Code
    ergebnis = []
    for r in range(n):
        text = ''
        for c in range(n):
            if len(moeglich[r][c]) == 2:
                text = text + '? '
            else:
                text = text + str(erste[r][c]) + ' '
        ergebnis.append(text)
    return ergebnis

for name in ['treppe', 'ring', 'unmoeglich']:
    daten = lies_tomograph('daten/raster_' + name + '.txt')
    n = daten[0][0]
    linien, soll = baue_linien(n, daten[1], daten[2])
    aufrufe[0] = 0
    ergebnis = fragezeichen(n, linien, soll)
    print(name, ':', aufrufe[0], 'Aufrufe')
    if ergebnis is None:
        print('keine passende Figur')
    else:
        for text in ergebnis:
            print(text)

<details>
<summary>Tipp anzeigen</summary>

Drei Schleifen: über `r`, `c` und `wert` in `[0, 1]`. Ist `wert` nicht in `moeglich[r][c]`, ein ganz offenes Raster bauen,
`versuch[r][c] = wert` setzen und `suche_figur` aufrufen. Ist das Ergebnis nicht `None`, für alle `i, j`
`moeglich[i][j].add(figur[i][j])`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def fragezeichen(n, linien, soll):
    erste = suche_figur([[OFFEN] * n for r in range(n)], linien, soll)
    if erste is None:
        return None
    moeglich = []
    for r in range(n):
        zeile = []
        for c in range(n):
            zeile.append({erste[r][c]})
        moeglich.append(zeile)
    for r in range(n):
        for c in range(n):
            for wert in [0, 1]:
                if wert not in moeglich[r][c]:
                    versuch = [[OFFEN] * n for i in range(n)]
                    versuch[r][c] = wert                     # dieses Feld festlegen
                    figur = suche_figur(versuch, linien, soll)
                    if figur is not None:
                        for i in range(n):                   # alle Werte der Figur vermerken
                            for j in range(n):
                                moeglich[i][j].add(figur[i][j])
    ergebnis = []
    for r in range(n):
        text = ''
        for c in range(n):
            if len(moeglich[r][c]) == 2:
                text = text + '? '
            else:
                text = text + str(erste[r][c]) + ' '
        ergebnis.append(text)
    return ergebnis

for name in ['treppe', 'ring', 'unmoeglich']:
    daten = lies_tomograph('daten/raster_' + name + '.txt')
    n = daten[0][0]
    linien, soll = baue_linien(n, daten[1], daten[2])
    aufrufe[0] = 0
    ergebnis = fragezeichen(n, linien, soll)
    print(name, ':', aufrufe[0], 'Aufrufe')
    if ergebnis is None:
        print('keine passende Figur')
    else:
        for text in ergebnis:
            print(text)
```

Beim Ring sind alle 16 Felder des inneren 4 × 4-Quadrats unsicher. Tatsächlich gibt es 90 Figuren mit diesen Summen. Zeilen-
und Spaltensummen allein verraten also oft wenig. Darum schaut Azze auch diagonal.

</details>

### Für Schnelle: mit Diagonalen

In der echten Aufgabe kommen zwei Zeilen dazu: die Summen der **blauen** Diagonalen (wie „/“, Feld `(r, c)` liegt auf Linie
`r + c`) und der **gelben** Diagonalen (wie „\\“, Linie `c - r + n - 1`). Es gibt von jeder Sorte `2n - 1` Diagonalen. Die
Dateien `tomograph00.txt`, `tomograph03.txt` und `tomograph05.txt` sind drei Beispiele der echten Aufgabe (00 ist das Bild
vom Aufgabenblatt).

#### Aufgabe E.5

Schreibe `ergaenze_diagonalen(n, blau, gelb, linien, soll)`, die die Diagonalen an `linien` und `soll` anhängt. Werte die drei
Beispiele einmal nur mit Zeilen und Spalten und einmal mit Diagonalen aus.

**Erwartete Ausgabe:**
```
tomograph00 : nur Zeilen und Spalten 64 Fragezeichen, 504 Aufrufe
    mit Diagonalen, 68 Aufrufe:
    0 0 1 1 1 0 0 0 
    0 0 1 0 1 0 0 0 
    0 0 1 1 1 0 0 0 
    1 0 0 1 0 0 0 0 
    0 1 1 1 1 1 1 0 
    0 0 0 1 0 0 1 0 
    0 0 0 1 0 0 0 1 
    0 0 1 0 1 0 0 0 
tomograph03 : nur Zeilen und Spalten 16 Fragezeichen, 28 Aufrufe
    mit Diagonalen, 11 Aufrufe:
    0 ? ? 1 
    ? 1 1 ? 
    ? 0 0 ? 
    0 ? ? 1 
tomograph05 : nur Zeilen und Spalten 36 Fragezeichen, 238 Aufrufe
    mit Diagonalen, 26 Aufrufe:
    0 ? ? ? ? 0 
    ? ? 1 ? ? ? 
    ? 1 1 1 ? ? 
    ? ? 1 1 ? ? 
    ? ? ? ? ? 0 
    0 ? ? ? 0 1
```

In [ ]:
def ergaenze_diagonalen(n, blau, gelb, linien, soll):
    pass                    # dein Code

for nummer in ['00', '03', '05']:
    daten = lies_tomograph('daten/tomograph' + nummer + '.txt')
    n = daten[0][0]
    linien, soll = baue_linien(n, daten[1], daten[2])
    aufrufe[0] = 0
    ohne = fragezeichen(n, linien, soll)
    fragen = 0
    for text in ohne:
        fragen = fragen + text.count('?')
    print('tomograph' + nummer, ': nur Zeilen und Spalten', fragen, 'Fragezeichen,', aufrufe[0], 'Aufrufe')
    ergaenze_diagonalen(n, daten[3], daten[4], linien, soll)
    aufrufe[0] = 0
    mit = fragezeichen(n, linien, soll)
    print('    mit Diagonalen,', aufrufe[0], 'Aufrufe:')
    for text in mit:
        print('   ', text)

<details>
<summary>Tipp anzeigen</summary>

Für die blaue Linie `d` gehst du alle Zeilen `r` durch und nimmst `c = d - r`, wenn `0 <= c < n` ist. Für die gelbe Linie `d`
ist `c = d + r - n + 1`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def ergaenze_diagonalen(n, blau, gelb, linien, soll):
    for d in range(2 * n - 1):              # blaue Linie d: alle Felder mit r + c == d
        linie = []
        for r in range(n):
            c = d - r
            if 0 <= c < n:
                linie.append((r, c))
        linien.append(linie)
        soll.append(blau[d])
    for d in range(2 * n - 1):              # gelbe Linie d: alle Felder mit c - r + n - 1 == d
        linie = []
        for r in range(n):
            c = d + r - n + 1
            if 0 <= c < n:
                linie.append((r, c))
        linien.append(linie)
        soll.append(gelb[d])

for nummer in ['00', '03', '05']:
    daten = lies_tomograph('daten/tomograph' + nummer + '.txt')
    n = daten[0][0]
    linien, soll = baue_linien(n, daten[1], daten[2])
    aufrufe[0] = 0
    ohne = fragezeichen(n, linien, soll)
    fragen = 0
    for text in ohne:
        fragen = fragen + text.count('?')
    print('tomograph' + nummer, ': nur Zeilen und Spalten', fragen, 'Fragezeichen,', aufrufe[0], 'Aufrufe')
    ergaenze_diagonalen(n, daten[3], daten[4], linien, soll)
    aufrufe[0] = 0
    mit = fragezeichen(n, linien, soll)
    print('    mit Diagonalen,', aufrufe[0], 'Aufrufe:')
    for text in mit:
        print('   ', text)
```

Mit den Diagonalen ist das Bild vom Aufgabenblatt eindeutig, ohne sie wären alle 64 Felder unsicher. `tomograph03` und
`tomograph05` bleiben mehrdeutig (laut einer vollständigen Zählung 2 und 6 passende Figuren). Die Fragezeichen stehen an
denselben Stellen wie im Lösungsnotebook. Mehr Linien bedeuten mehr Schlüsse: Die Suche braucht weniger Aufrufe, obwohl
jedes Schließen mehr Linien prüft.

</details>

#### Frage E.6

Für die Dokumentation: Warum ist ein Fragezeichen sicher richtig, und warum ist ein Feld **ohne** Fragezeichen sicher
eindeutig?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Ein Fragezeichen steht nur, wenn zwei **gefundene** Figuren an dieser Stelle verschiedene Werte haben. Beide passen zu allen
Summen, denn `suche_figur` gibt nur Figuren zurück, bei denen das Schließen keinen Widerspruch fand und kein Feld offen
blieb. Also ist das Feld wirklich nicht eindeutig.

Hat ein Feld kein Fragezeichen, wurde für den anderen Wert eine vollständige Suche gestartet, und sie fand nichts. Die Suche
ist vollständig, weil die Regeln keine Figur ausschließen (sie folgern nur, was zwingend ist) und das Raten beide Werte
probiert. Also gibt es keine passende Figur mit dem anderen Wert.

</details>

## Das hast du gelernt

- **Kandidaten** eines Feldes bestimmen: Was ist hier noch möglich?
- **Regeln** liefern zwingende Folgerungen. Sie werden wiederholt, **bis sich nichts mehr ändert** (`while geaendert`).
  Eine Regel darf nur eintragen, was in **jeder** Lösung so sein muss.
- **Widersprüche** (kein Kandidat, kein Platz, Summe nicht erreichbar) beenden einen Versuch sofort.
- **Erst schließen, dann raten:** nach jedem Raten wieder schließen. Mit einer **Kopie** je Aufruf statt Zurücknehmen, und
  bei Listen von Listen Zeile für Zeile kopieren.
- **Eindeutigkeit:** Lösungen bis zur Grenze 2 sammeln. Für Fragezeichen je Feld beide Werte prüfen und gefundene Figuren
  für alle Felder vermerken.
- **Drei Zustände** je Feld: ja, nein, offen.
- **Gleich modellieren:** Zeilen, Spalten, Diagonalen als Linien mit Soll-Anzahl, dann gilt eine Regel für alle.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Bibertomograph](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/44_Bibertomograph/bibertomograph_loesung.ipynb) (44. BwInf): alle Beispiele bis 12 × 12,
  Auswertung als Bild
- [Zimmerbelegung](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/36_Zimmerbelegung/zimmerbelegung_loesung.ipynb) (36.)
- [Fehlerfrei puzzeln](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/30_FehlerfreiPuzzeln/fehlerfreipuzzeln_loesung.ipynb) (30.)
- [Sudokopie](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/41_Sudokopie/sudokopie_loesung.ipynb) (41.): Sudokus, die nur umgeordnet sind, erkennen

Im nächsten Notebook geht es um **Breitensuche**: kürzeste Wege in Labyrinthen und Netzen.